# Imports

In [57]:
import pandas as pd
import numpy as df
import csv
import html
import re
import contractions
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Dataset Preparation

## 1(a)

In [40]:
raw_data = pd.read_csv('data.tsv', sep='\t', dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False, on_bad_lines='skip', usecols=['review_body', 'star_rating'])
print(len(raw_data))
print(raw_data['star_rating'].value_counts().head(10))
print((raw_data['review_body'].str.strip() == '').sum())

2642434
star_rating
5    1584192
4     418694
1     307234
3     193818
2     138496
Name: count, dtype: int64
80


In [41]:
df = raw_data.rename(columns={'review_body': 'review', 'star_rating': 'rating'})[['review', 'rating']]
ratings = pd.to_numeric(df['rating'], errors='coerce')
valid_rating = ratings.isin([1, 2, 3, 4, 5])
valid_review = df['review'].str.strip() != ''
valid = valid_rating & valid_review

data = df.loc[valid, ['review']].copy()
data['rating'] = ratings[valid].astype(int)

print(int((~valid).sum()))
print(len(data))
print(data["rating"].value_counts().sort_index())

80
2642354
rating
1     307223
2     138493
3     193813
4     418683
5    1584142
Name: count, dtype: int64


## 1(b)

In [42]:
rating_counts = {}

for r in range(1,6):
    is_r = data['rating'] == r
    count = is_r.sum()
    rating_counts[r] = int(count)

class_counts = {
    "positive": rating_counts[4] + rating_counts[5],
    "negative": rating_counts[1] + rating_counts[2],
    "neutral":  rating_counts[3],
}
print(class_counts)

{'positive': 2002825, 'negative': 445716, 'neutral': 193813}


In [43]:
data = data[data["rating"] != 3].copy()
data["label"] = (data["rating"] >= 4).astype(int)

positive_data = data[data["label"] == 1].sample(n=100000, replace=False, random_state=42)
negative_data = data[data["label"] == 0].sample(n=100000, replace=False, random_state=42)

print(len(data))
print(len(positive_data), len(negative_data))
print(positive_data['rating'].value_counts().sort_index())

2448541
100000 100000
rating
4    21003
5    78997
Name: count, dtype: int64


## 1(c)

In [44]:
sampled_data = pd.concat([positive_data, negative_data]).reset_index(drop=True)
train_df, test_df = train_test_split(sampled_data, test_size=0.2, stratify=sampled_data['label'], random_state=42)

print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().to_dict(), test_df["label"].value_counts().to_dict())
print(train_df.head())

(160000, 3) (40000, 3)
{0: 80000, 1: 80000} {1: 20000, 0: 20000}
                                                   review  rating  label
181188  We've been using the LC-51 ink for our brother...       1      0
194598                       DO NOT BUY, BOX IS TOO SMALL       1      0
5172    I'm a long time fan of Uni-ball in general and...       5      1
196404  I've had Neat Receipts since version 2.0.  The...       1      0
73301   Nice to find these at Amazon, since my local o...       5      1


In [45]:
def prepare_data(path, n_per_class=100000):
    """Return (train_df, test_df, stats), as specified in Section 1/interface."""
    raw_data = pd.read_csv(path, sep='\t', dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False, on_bad_lines='skip', usecols=['review_body', 'star_rating'])
    df = raw_data.rename(columns={'review_body': 'review', 'star_rating': 'rating'})[['review', 'rating']]

    ratings = pd.to_numeric(df['rating'], errors='coerce')
    valid_rows = ratings.isin([1, 2, 3, 4, 5]) & (df['review'].str.strip() != '')
    excluded_rows = int((~valid_rows).sum())

    data = df.loc[valid_rows, ['review']].copy()
    data['rating'] = ratings[valid_rows].astype(int)

    rating_counts = {}

    for r in range(1, 6):
        is_r = data['rating'] == r
        count = is_r.sum()
        rating_counts[r] = int(count)

    class_counts = {
        "positive": rating_counts[4] + rating_counts[5],
        "negative": rating_counts[1] + rating_counts[2],
        "neutral": rating_counts[3],
    }

    data = data[data["rating"] != 3].copy()
    data["label"] = (data["rating"] >= 4).astype(int)

    positive_data = data[data["label"] == 1].sample(n=n_per_class, replace=False, random_state=42)
    negative_data = data[data["label"] == 0].sample(n=n_per_class, replace=False, random_state=42)

    sampled_data = pd.concat([positive_data, negative_data]).reset_index(drop=True)
    train_df, test_df = train_test_split(sampled_data, test_size=0.2, stratify=sampled_data['label'], random_state=42)

    stats = {'rating_counts': rating_counts,
             'class_counts': class_counts,
             'excluded_rows': excluded_rows}

    return train_df, test_df, stats

In [46]:
train_df, test_df, stats = prepare_data("data.tsv")
print(stats["class_counts"], stats["excluded_rows"])
print(train_df.index[:5].tolist())   # expect [181188, 194598, 5172, 196404, 73301]

small_train, small_test, _ = prepare_data("data.tsv", n_per_class=50)
print(small_train.shape, small_test.shape)   # expect (80, 3) (20, 3)

{'positive': 2002825, 'negative': 445716, 'neutral': 193813} 80
[181188, 194598, 5172, 196404, 73301]
(80, 3) (20, 3)


# Data Cleaning

In [47]:
def clean_review(text):
    text = html.unescape(text).lower()
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = contractions.fix(text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return ' '.join(text.split())

In [48]:
print(clean_review("These pens aren’t reliable."))     # note: curly apostrophe
print(clean_review("&lt;p&gt;Great!!!&lt;/p&gt;"))
print(clean_review("😀 10/10"))

these pens are not reliable
great



In [49]:
train_df['clean'] = train_df['review'].apply(clean_review)
test_df['clean'] = test_df['review'].apply(clean_review)

print(f"Average length before cleaning: {train_df['review'].str.len().mean():.4f}")
print(f"Average length after cleaning: {train_df['clean'].str.len().mean():.4f}")
print((train_df["clean"] == "").sum())   # how many became empty?

Average length before cleaning: 318.8318
Average length after cleaning: 303.7834
57


# Preprocessing

In [50]:
from nltk.corpus import stopwords

sw = stopwords.words("english")
print(len(sw))
print(sw[:40])
print([w for w in ["no", "nor", "not", "very", "but", "against", "don", "t"] if w in sw])

198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don']
['no', 'nor', 'not', 'very', 'but', 'against', 'don', 't']


In [51]:
from nltk.stem import WordNetLemmatizer

L = WordNetLemmatizer()
for w in ["pens", "batteries", "mice", "boxes", "working", "printed", "better", "us", "glasses", "less"]:
    print(f"{w:10} -> {L.lemmatize(w)}")

pens       -> pen
batteries  -> battery
mice       -> mouse
boxes      -> box
working    -> working
printed    -> printed
better     -> better
us         -> u
glasses    -> glass
less       -> less


In [54]:
# Lazy loading
_STOP_WORDS = None
_LEMMATIZER = None

def preprocess_review(text):
    global _STOP_WORDS, _LEMMATIZER
    if _STOP_WORDS is None:
        _STOP_WORDS = set(stopwords.words('english')) - {'no', 'nor', 'not'}
        _LEMMATIZER = WordNetLemmatizer()

    tokens = []
    for word in text.split():
        if not word in _STOP_WORDS:
            tokens.append(word)

    lemmas = []
    for word in tokens:
        lemmas.append(_LEMMATIZER.lemmatize(word))

    return ' '.join(lemmas)

In [55]:
print(repr(preprocess_review(clean_review("These pens aren’t reliable."))))

'pen not reliable'


In [56]:
train_df["prep"] = train_df["clean"].apply(preprocess_review)
test_df["prep"] = test_df["clean"].apply(preprocess_review)

print(f"Average length before preprocessing: {train_df['clean'].str.len().mean():.4f}")
print(f"Average length after preprocessing: {train_df['prep'].str.len().mean():.4f}")
print((train_df["prep"] == "").sum())

Average length before preprocessing: 303.7834
Average length after preprocessing: 191.9631
133


# Feature Extraction

In [58]:
def extract_features(train_texts, test_texts):
    vectorizer = TfidfVectorizer()
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return vectorizer, X_train, X_test

In [59]:
vectorizer, X_train, X_test = extract_features(train_df["prep"].tolist(), test_df["prep"].tolist())

print(f"TF-IDF matrix shape: {X_train.shape}")
print(X_test.shape)
print(type(X_train))
print(X_train.nnz / (X_train.shape[0] * X_train.shape[1]))   # fraction of nonzero cells

TF-IDF matrix shape: (160000, 50609)
(40000, 50609)
<class 'scipy.sparse._csr.csr_matrix'>
0.0004760597176391551


# Model Training and Evaluation

In [60]:
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def train_model(name, X_train, y_train):
    """Return a fitted classifier for one of the three specified model names."""
    if name == 'Perceptron':
        model = Perceptron(random_state=42, max_iter=1000, tol=1e-3)
    elif name == 'Logistic Regression':
        model = LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000, tol=1e-4)
    elif name == 'Naive Bayes':
        model = MultinomialNB(alpha=1.0)

    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X, y):
    """Return unrounded accuracy, precision, recall, f1 in a dictionary."""
    y_prediction = model.predict(X)
    accuracy = float(accuracy_score(y, y_prediction))
    precision = float(precision_score(y, y_prediction, average='binary', pos_label=1, zero_division=0))
    recall = float(recall_score(y, y_prediction, average='binary', pos_label=1, zero_division=0))
    f1 = float(f1_score(y, y_prediction, average='binary', pos_label=1, zero_division=0))

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [61]:
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

for name in ["Perceptron", "Logistic Regression", "Naive Bayes"]:
    model = train_model(name, X_train, y_train)
    tr = evaluate_model(model, X_train, y_train)
    te = evaluate_model(model, X_test, y_test)
    for split, m in [("Training", tr), ("Testing", te)]:
        print(f"{name} {split} Accuracy: {m['accuracy']:.4f}")
        print(f"{name} {split} Precision: {m['precision']:.4f}")
        print(f"{name} {split} Recall: {m['recall']:.4f}")
        print(f"{name} {split} F1-score: {m['f1']:.4f}")

Perceptron Training Accuracy: 0.9119
Perceptron Training Precision: 0.9371
Perceptron Training Recall: 0.8832
Perceptron Training F1-score: 0.9093
Perceptron Testing Accuracy: 0.8717
Perceptron Testing Precision: 0.8940
Perceptron Testing Recall: 0.8434
Perceptron Testing F1-score: 0.8680
Logistic Regression Training Accuracy: 0.9206
Logistic Regression Training Precision: 0.9238
Logistic Regression Training Recall: 0.9168
Logistic Regression Training F1-score: 0.9203
Logistic Regression Testing Accuracy: 0.9113
Logistic Regression Testing Precision: 0.9134
Logistic Regression Testing Recall: 0.9087
Logistic Regression Testing F1-score: 0.9111
Naive Bayes Training Accuracy: 0.8797
Naive Bayes Training Precision: 0.8920
Naive Bayes Training Recall: 0.8639
Naive Bayes Training F1-score: 0.8777
Naive Bayes Testing Accuracy: 0.8685
Naive Bayes Testing Precision: 0.8798
Naive Bayes Testing Recall: 0.8536
Naive Bayes Testing F1-score: 0.8665


# Bigram Model

In [65]:
from collections import Counter

def fit_bigram(train_texts, min_count=3):
    word_counts = Counter()
    for text in train_texts:
        for word in text.split():
            word_counts[word] += 1

    kept_words = set() # keep words seen at least min_count
    for word, count in word_counts.items():
        if count >= min_count:
            kept_words.add(word)
    vocab = kept_words | {'<unk>', '</s>'}

    bigrams = Counter()
    contexts = Counter()

    for text in train_texts:
        seq = ['<s>']
        for word in text.split():
            if word in kept_words:
                seq.append(word)
            else:
                seq.append('<unk>')
        seq.append('</s>')

        for i in range(len(seq) - 1):
            context, word = seq[i], seq[i + 1]
            bigrams[(context, word)] += 1
            contexts[context] += 1

    return {
        'vocab': vocab,
        'bigrams': bigrams,
        'contexts': contexts
    }

In [66]:
m = fit_bigram(["the pen is great", "the pen broke", "great pen"], min_count=2)
print(sorted(m["vocab"]), len(m["vocab"]))
print(m["bigrams"][("pen", "<unk>")], m["contexts"]["pen"])   # expect 2 3

['</s>', '<unk>', 'great', 'pen', 'the'] 5
2 3


In [67]:
lm_train_pool = train_df.loc[train_df["clean"] != "", "clean"]
lm_test_pool = test_df.loc[test_df["clean"] != "", "clean"]

lm_train = lm_train_pool.sample(n=5000, replace=False, random_state=42).tolist()
lm_test = lm_test_pool.sample(n=1000, replace=False, random_state=42).tolist()

bigram_model = fit_bigram(lm_train)
print(f"Bigram vocabulary size: {len(bigram_model['vocab'])}")

Bigram vocabulary size: 4957


In [71]:
import math

def evaluate_bigram(model, test_texts, k=1.0):
    vocab = model['vocab']
    bigrams = model['bigrams']
    contexts = model['contexts']
    vocab_len = len(vocab)
    kept = vocab - {'<unk>', '</s>'}

    unseen = 0
    n_tokens = 0
    log_probability_sum = 0.0

    for text in test_texts:
        sequence = ['<s>']
        for word in text.split():
            if word in kept:
                sequence.append(word)
            else:
                sequence.append('<unk>')
        sequence.append('</s>')

        for i in range(len(sequence) - 1):
            context, word = sequence[i], sequence[i + 1]
            c_hw = bigrams.get((context, word), 0)
            c_h = contexts.get(context, 0)
            if c_hw == 0:
                unseen += 1
            probability = (c_hw + k) / (c_h + k * vocab_len)
            log_probability_sum += math.log(probability)
            n_tokens += 1

    perplexity = math.exp(-log_probability_sum / n_tokens)
    return {
        'unseen': unseen,
        'n_tokens': n_tokens,
        'perplexity': perplexity
    }


In [72]:
print(evaluate_bigram(m, ["the pen", "pen broke the"]))           # unseen 3, n_tokens 7, PP ≈ 4.2732
print(evaluate_bigram(m, ["the pen", "pen broke the"], k=0.1))    # same counts, PP ≈ 5.8345

{'unseen': 3, 'n_tokens': 7, 'perplexity': 4.273182809374539}
{'unseen': 3, 'n_tokens': 7, 'perplexity': 5.834530946891399}


In [73]:
r1 = evaluate_bigram(bigram_model, lm_test, k=1.0)
r2 = evaluate_bigram(bigram_model, lm_test, k=0.1)
r3 = evaluate_bigram(bigram_model, lm_test, k=0.01)

print(f"Bigram vocabulary size: {len(bigram_model['vocab'])}")
print(f"Bigram unseen test transitions: {r1['unseen']}")
print(f"Bigram test perplexity: {r1['perplexity']:.4f}")
print(f"Bigram test perplexity (k=0.1): {r2['perplexity']:.4f}")
print(f"Bigram test perplexity (k=0.01): {r3['perplexity']:.4f}")
print(r1["n_tokens"], r1["unseen"] / r1["n_tokens"])   # extra: N and the unseen fraction

Bigram vocabulary size: 4957
Bigram unseen test transitions: 14368
Bigram test perplexity: 580.1261
Bigram test perplexity (k=0.1): 276.4948
Bigram test perplexity (k=0.01): 231.4060
66295 0.21672826004977752


# Run Experiment

In [74]:
def average_len(texts):
    total = 0
    for t in texts:
        total += len(t)

    return total / len(texts)

def run_experiment(path="data.tsv", n_per_class=100000,
                   lm_train_size=5000, lm_test_size=1000):

    # Section 1
    train_df, test_df, stats = prepare_data(path, n_per_class)
    counts = stats['class_counts']
    print(f'Positive reviews: {counts["positive"]}')
    print(f'Negative reviews: {counts["negative"]}')
    print(f'Neutral reviews: {counts["neutral"]}')

    # Section 2 and 3
    train_raw = train_df['review'].tolist()
    test_raw = test_df['review'].tolist()

    train_clean = []
    for text in train_raw:
        train_clean.append(clean_review(text))

    test_clean = []
    for text in test_raw:
        test_clean.append(clean_review(text))

    train_prep = []
    for text in train_clean:
        train_prep.append(preprocess_review(text))

    test_prep = []
    for text in test_clean:
        test_prep.append(preprocess_review(text))

    print(f'Average length before cleaning: {average_len(train_raw):.4f}')
    print(f'Average length after cleaning: {average_len(train_clean):.4f}')
    print(f'Average length before preprocessing: {average_len(train_clean):.4f}')
    print(f'Average length after preprocessing: {average_len(train_prep):.4f}')

    # Section 4 to 7
    _, X_train, X_test = extract_features(train_prep, test_prep)
    print(f'TF-IDF matrix shape: {X_train.shape}')

    y_train = train_df['label'].to_numpy()
    y_test = test_df['label'].to_numpy()

    for name in ['Perceptron', 'Logistic Regression', 'Naive Bayes']:
        model = train_model(name, X_train, y_train)

        train_metrics = evaluate_model(model, X_train, y_train)
        print(f'{name} Training Accuracy: {train_metrics["accuracy"]:.4f}')
        print(f'{name} Training Precision: {train_metrics["precision"]:.4f}')
        print(f'{name} Training Recall: {train_metrics["recall"]:.4f}')
        print(f'{name} Training F1-score: {train_metrics["f1"]:.4f}')

        test_metrics = evaluate_model(model, X_test, y_test)
        print(f'{name} Testing Accuracy: {test_metrics["accuracy"]:.4f}')
        print(f'{name} Testing Precision: {test_metrics["precision"]:.4f}')
        print(f'{name} Testing Recall: {test_metrics["recall"]:.4f}')
        print(f'{name} Testing F1-score: {test_metrics["f1"]:.4f}')

    # Section 8
    train_clean_series = pd.Series(train_clean, index=train_df.index)
    test_clean_series = pd.Series(test_clean, index=test_df.index)
    language_model_train = train_clean_series[train_clean_series != ""].sample(n=lm_train_size, replace=False, random_state=42).tolist()
    language_model_test = test_clean_series[test_clean_series != ""].sample(n=lm_test_size, replace=False, random_state=42).tolist()

    bigram_model = fit_bigram(language_model_train)

    results = {}
    for k in (1.0, 0.1, 0.01):
        results[k] = evaluate_bigram(bigram_model, language_model_test, k=k)

    print(f'Bigram vocabulary size: {len(bigram_model["vocab"])}')
    print(f'Bigram unseen test transitions: {results[1.0]["unseen"]}')
    print(f'Bigram test perplexity: {results[1.0]["perplexity"]:.4f}')
    print(f'Bigram test perplexity (k=0.1): {results[0.1]["perplexity"]:.4f}')
    print(f'Bigram test perplexity (k=0.01): {results[0.01]["perplexity"]:.4f}')

In [75]:
run_experiment()

Positive reviews: 2002825
Negative reviews: 445716
Neutral reviews: 193813
Average length before cleaning: 318.8318
Average length after cleaning: 303.7834
Average length before preprocessing: 303.7834
Average length after preprocessing: 191.9631
TF-IDF matrix shape: (160000, 50609)
Perceptron Training Accuracy: 0.9119
Perceptron Training Precision: 0.9371
Perceptron Training Recall: 0.8832
Perceptron Training F1-score: 0.9093
Perceptron Testing Accuracy: 0.8717
Perceptron Testing Precision: 0.8940
Perceptron Testing Recall: 0.8434
Perceptron Testing F1-score: 0.8680
Logistic Regression Training Accuracy: 0.9206
Logistic Regression Training Precision: 0.9238
Logistic Regression Training Recall: 0.9168
Logistic Regression Training F1-score: 0.9203
Logistic Regression Testing Accuracy: 0.9113
Logistic Regression Testing Precision: 0.9134
Logistic Regression Testing Recall: 0.9087
Logistic Regression Testing F1-score: 0.9111
Naive Bayes Training Accuracy: 0.8797
Naive Bayes Training Preci